# Publish Triangle Mesh from Multipatch Shapefile

This notebook shows how you can sign in and publish a Triangle Mesh geoscience object from a shapefile containing only multipatch shapes to your chosen Evo workspace.

**Important:** This notebook requires Python 3.10, 3.11, or 3.12 (not 3.14+) due to evo.notebooks dependencies.

In the first cell we create a ServiceManagerWidget which will open a browser window and ask you to sign in.

Once signed in, a widget will be displayed to allow you to select your organisation and an Evo workspace.

__Required:__ In Cell 2, replace `"your-client-id"` with your Evo app client ID before running the cell.

In [1]:
from evo.notebooks import ServiceManagerWidget

client_id = "core-compute-tasks-notebooks"
redirect_url = "http://localhost:3000/signin-callback"
ims_base_uri = "https://qa-ims.bentley.com"
discovery_base_uri = "https://int-discover.test.api.seequent.com"

manager = await ServiceManagerWidget.with_auth_code(
        client_id=client_id,
        base_uri=ims_base_uri,
        discovery_url=discovery_base_uri,
        redirect_url=redirect_url,
    ).login()

C:\Users\Dawson.Brown\Documents\Programming\evo-data-converters\.venv\lib\site-packages\evo\notebooks\authorizer.py:108: UserWarning: The evo.notebooks.AuthorizationCodeAuthorizer is not secure, and should only ever be used in Jupyter notebooks in a private environment.
  warnings.warn(


ServiceManagerWidget(children=(VBox(children=(HBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR…

## Select Organization, Hub, and Workspace

This cell prepares your active Evo context for publishing.

It will:
- List available organizations and select one (by `org_index`)
- List hubs and select one (by `hub_index`)
- List workspaces and select one (by `workspace_index`)
- Create a new workspace only if none exist

Before running, update the index values to match the organization, hub, and workspace you want to use.

In [2]:
# Select organization and workspace, or create one if needed
service_mgr = manager._service_manager

# Variables to store workspace info
new_workspace = None
selected_workspace_id = None

# List available organizations
orgs = list(service_mgr.list_organizations())
print("Available Organizations:")
for i, org in enumerate(orgs):
    print(f"  {i}: {org.display_name} (ID: {org.id})")

if not orgs:
    print("  No organizations found!")
else:
    # Select your organization by index (change the number if needed)
    org_index = 0  # Change this to the index of your organization
    service_mgr.set_current_organization(orgs[org_index].id)
    print(f"\n✅ Selected organization: {orgs[org_index].display_name}")

    # List available hubs
    hubs = list(service_mgr.list_hubs())
    print("\nAvailable Hubs:")
    for i, hub in enumerate(hubs):
        print(f"  {i}: {hub.display_name} (Code: {hub.code})")

    if hubs:
        # Select your hub by index (change the number if needed)
        hub_index = 0  # Change this: 0=Australia East, 1=Canada Central, 2=South Africa North and so on...
        service_mgr.set_current_hub(hubs[hub_index].code)
        print(f"\n✅ Selected hub: {hubs[hub_index].display_name}")

        # List available workspaces
        workspaces = list(service_mgr.list_workspaces())
        print("\nAvailable Workspaces:")

        if workspaces:
            for i, ws in enumerate(workspaces):
                print(f"  {i}: {ws.display_name} (ID: {ws.id})")

            # Select your workspace by index (change the number if needed)
            workspace_index = 0  # Change this to the index of your workspace
            service_mgr.set_current_workspace(workspaces[workspace_index].id)
            selected_workspace_id = workspaces[workspace_index].id
            print(f"\n✅ Selected workspace: {workspaces[workspace_index].display_name}")
        else:
            print("  No workspaces found!")
            print("\n💡 Creating a new workspace...")

            # Create a workspace
            from datetime import datetime

            from evo.workspaces import WorkspaceAPIClient

            connector = manager.get_connector()
            workspace_client = WorkspaceAPIClient(connector, orgs[org_index].id)

            # Create workspace with timestamp
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            workspace_name = f"Shapefile Workspace {timestamp}"

            try:
                new_workspace = await workspace_client.create_workspace(
                    name=workspace_name, description="Workspace for Shapefile to Mesh conversion"
                )

                selected_workspace_id = new_workspace.id
                print(f"✅ Workspace created: {new_workspace.display_name}")
                print(f"   Workspace ID: {new_workspace.id}")

                from evo.service_manager.manager import _State

                current_workspaces = list(service_mgr.list_workspaces())
                current_workspaces.append(new_workspace)

                service_mgr._ServiceManager__state = _State(
                    organizations=list(service_mgr.list_organizations()),
                    workspaces=current_workspaces,
                    selected_org_id=orgs[org_index].id,
                    selected_hub_code=hubs[hub_index].code,
                    selected_workspace_id=new_workspace.id,
                )

                print("✅ Workspace selected and ready for publishing!")

            except Exception as e:
                print(f"❌ Failed to create workspace: {e}")
                import traceback

                traceback.print_exc()
    else:
        print("  No hubs found!")

Available Organizations:
  0: Bentley Systems Inc on Hub Integration Canada Central(351mt) (ID: d844342d-a544-4a0b-b146-fc35036edf38)

✅ Selected organization: Bentley Systems Inc on Hub Integration Canada Central(351mt)

Available Hubs:
  0: Integration Canada Central (Code: 351mt)

✅ Selected hub: Integration Canada Central

Available Workspaces:
  0: Dawson_Test (ID: ba30fdbd-5e79-40a7-9318-30e1a6eec58e)

✅ Selected workspace: Dawson_Test


## Convert and Publish Shapefile to Triangular Mesh

In the cell below we:
1. Specify the path to our PNG file
2. Set the shapefile parameters (in particular the epsg code)
3. Add optional tags
4. Call `convert_shp` to convert and publish the image

The shapefile will be:
- Translated to vertices, triangles, and shapes which are stored in parquet files
- Formatted according to the triangle-mesh schema
- Published to your evo workspace

**Note:** If you don't have workspaces available, use `evo_workspace_metadata` parameter instead of `service_manager_widget`.

In [3]:
# Install local package into this notebook kernel (run once per environment)
%pip install -e ../..

Obtaining file:///M:/evo-data-converters/packages/shp
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for evo-data-converters-shp (pyproject.toml): started
  Building editable for evo-data-converters-shp (pyproject.toml): finished with status 'done'
  Created wheel for evo-data-converters-shp: filename=evo_data_converters_shp-0.1.0-py3-none-any.whl size=7358 sha256=f206abef6ec2c4b6a1c1aafa6f59d13c63d9e487a70a9dc9534f0a8f8c

In [4]:
from pathlib import Path

from evo.data_converters.shp.importer.shp_to_evo import convert_shp

# Path to your dhapefile (can be any of the individual files or just the extensionless name)
# Using relative path from notebook location
filename = "../../tests/data/test_shape.shp"

# Coordinate Reference System (optional)
epsg_code = "EPSG:4326"

# Tags to add to the geoscience object
tags = {"Source": "Jupyter Notebook", "Type": "Shapefile Triangular Mesh"}

# Upload path in Evo workspace
upload_path = "shapefile/imports"

print("Converting and publishing image to Evo workspace...")
print(f"Shapefile: {Path(filename).name}")
print(f"Workspace: {selected_workspace_id}")

# Convert and publish
results = convert_shp(
    filepath=filename,
    epsg_code=epsg_code,
    tags=tags,
    evo_workspace_metadata=None,
    service_manager_widget=manager,
    upload_path=upload_path,
    publish_objects=True,
    overwrite_existing_objects=True,
)

# Print results
print("\n✅ Successfully published!")
print("\nPublished objects:")
for obj_metadata in results:
    print(f"  - {obj_metadata.name}")
    print(f"    ID: {obj_metadata.id}")

Converting and publishing image to Evo workspace...
Shapefile: test_shape.shp
Workspace: ba30fdbd-5e79-40a7-9318-30e1a6eec58e
Publishing Shapefile

✅ Successfully published!

Published objects:
  - test_shape.json
    ID: 05036b4c-c0e0-4b30-92b0-ebef9a11a3e2


## Convert Without Publishing (for testing)

You can also convert the Shapefile without publishing to inspect the resulting object:

In [5]:
import json

from evo.data_converters.shp.importer.shp_to_evo import convert_shp

# Convert but don't publish
shp_objects = convert_shp(
    filepath="../../tests/data/test_shape.shp",
    epsg_code="EPSG:4326",
    upload_path="./parquet_output",
    publish_objects=False,  # Don't publish
)

# Inspect the mesh object
shp = shp_objects[0]
print(f"Shape name: {shp.name}")

shp_dict = shp.as_dict()  # Use as_dict() instead of to_dict()
print("\nJSON representation:")
print(json.dumps(shp_dict, indent=2))

Shape name: test_shape

JSON representation:
{
  "triangles": {
    "vertices": {
      "data": "790619c3bfccb5c4b3a088672ab096813979eb0290de9e11afbef95f036a29cc",
      "length": 772,
      "width": 3,
      "data_type": "float64"
    },
    "indices": {
      "data": "a701993b1450db26e1cd141bc98ff5d949dc6096089020539da9cbee39c767b3",
      "length": 3080,
      "width": 3,
      "data_type": "uint64"
    }
  },
  "parts": {
    "attributes": [
      {
        "name": "name",
        "key": "2f2a90df0637f6283473b3aa52ed1d09d9b7864f289864263130cf3ef5342d96",
        "attribute_type": "string",
        "values": {
          "data": "2f2a90df0637f6283473b3aa52ed1d09d9b7864f289864263130cf3ef5342d96",
          "length": 2,
          "data_type": "string"
        }
      }
    ],
    "chunks": {
      "data": "5696a165c8957c16692705452d5d3cca514f2a3ca16e2ba6745e80ec6e575719",
      "length": 2,
      "width": 2,
      "data_type": "uint64"
    }
  },
  "name": "test_shape",
  "uuid": null,